# ProductIQ Phase 4 — Step 4.15: Hybrid Retrieval

Run BM25 and semantic retrieval for the same query, then inspect the deduplicated hybrid candidate pool with native scores and provenance (no score fusion).

In [ ]:
from pathlib import Path

from productiq.config.settings import get_settings
from productiq.database.engine import create_engine_from_settings
from productiq.representation.query_contract import build_query_representation
from productiq.retrieval import create_bm25_retriever_from_index_path
from productiq.retrieval.contracts import RetrievalMethod, RetrievalRequest
from productiq.retrieval.embedding_encoder import create_bge_small_en_v15_encoder
from productiq.retrieval.hybrid_retriever import create_hybrid_retriever
from productiq.retrieval.semantic_retriever import create_semantic_retriever_from_engine

query_text = "black Nike running shoes"
top_k = 20
request = RetrievalRequest(query=build_query_representation(query_text), top_k=top_k)

settings = get_settings()
engine = create_engine_from_settings(settings)
lexical = create_bm25_retriever_from_index_path(
    Path("resources/processed/bm25_lexical_index.pkl")
)
encoder = create_bge_small_en_v15_encoder(device="cpu", allow_cpu_fallback=True)
semantic = create_semantic_retriever_from_engine(engine, encoder)
hybrid = create_hybrid_retriever(lexical, semantic)

bm25_response = lexical.retrieve(request)
semantic_response = semantic.retrieve(request)
hybrid_response = hybrid.retrieve(request)

print("BM25 count:", bm25_response.candidate_count)
print("Semantic count:", semantic_response.candidate_count)
print("Hybrid metadata:", hybrid_response.metadata.model_dump() if hybrid_response.metadata else None)
print("Hybrid unique count:", hybrid_response.candidate_count)
print()
for row in hybrid_response.candidates[:8]:
    print(row.product_id, row.resolved_retrieval_methods(), row.bm25_score, row.vector_score, row.score)
engine.dispose()